# Customer Churn Analysis
A readable EDA walkthrough for the IBM Telco Customer Churn dataset.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
from preprocessing import load_dataset, clean_dataset
raw = load_dataset(ROOT / 'data')
data, audit = clean_dataset(raw)
data.head()

## Inspection and cleaning audit

In [ ]:
data.info()
display(data.describe(include='all').T)
audit

## Target distribution

In [ ]:
display(data['Churn'].value_counts().rename('customers'))
display(data['Churn'].value_counts(normalize=True).rename('share'))
sns.countplot(data=data, x='Churn', hue='Churn', palette='YlOrBr', legend=False)
plt.title('Customer churn distribution'); plt.show()

## Numerical feature analysis

In [ ]:
numeric_features = ['tenure', 'MonthlyCharges', 'TotalCharges']
display(data.groupby('Churn')[numeric_features].agg(['mean', 'median']).round(2))
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for axis, column in zip(axes, numeric_features):
    sns.boxplot(data=data, x='Churn', y=column, hue='Churn', palette='YlOrBr', legend=False, ax=axis)
    axis.set_title(f'{column} by churn')
plt.tight_layout(); plt.show()

## Categorical feature analysis

In [ ]:
categorical_features = ['Contract', 'InternetService', 'PaymentMethod', 'OnlineSecurity', 'TechSupport']
for column in categorical_features:
    churn_rates = data.groupby(column, dropna=False)['Churn'].apply(lambda values: (values == 'Yes').mean()).sort_values(ascending=False)
    display(churn_rates.rename(f'{column} churn rate').round(3))
    churn_rates.plot(kind='bar', color='#c76832', title=f'Churn rate by {column}', ylabel='churn rate', xlabel='')
    plt.xticks(rotation=25, ha='right'); plt.tight_layout(); plt.show()

## Reproducible training
Run `python ../src/train.py` from this notebook directory or the repository root to generate model comparison, metrics, and the saved pipeline. The training script is the single source of truth for model fitting and serialization.